# <font color="Green">**Notebook Purpose**</font>

This notebook produces three supplementary figures to support the response to **Reviewer 1's Comment 4** on clustering methodology. The reviewer asked for additional methodological detail and stability assessments around the choice of distance metric, number of clusters (k), and linkage method. These figures provide visual evidence that the original modeling decisions — Ward's linkage with squared Euclidean distance at k=40 — are well-justified.

**Figure A: Internal validity diagnostics for selecting k.** Side-by-side WCSS (elbow) and gap-statistic curves over k = 2..50 using Ward's linkage.

**Figure B: Sensitivity to k.** A 2×2 grid of patient-medication heatmaps showing the same Ward's-linkage clustering at k = 20, 30, 40, and 50.

**Figure C: Sensitivity to linkage method.** A 2×2 grid of patient-medication heatmaps showing the four classical agglomerative-linkage methods — single, complete, average, and Ward — all at k = 40.

---

### <font color="Red">Required Data</font>

1. **`patient_vectors.pkl`** — 120-dim binary trajectory vectors (12 six-month bins × 10 medication-class slots). Used as input to all clustering runs.
2. **`patient_bins.pkl`** — per-patient list of 12 medication sets, used to render the heatmaps.

Both files are produced by `MedicationTrajectoryRepresentations.ipynb`.

---

### Notes on computation

- The gap-statistic computation is the slow step (≈ 30 min on a typical machine). Results are cached to `/content/cache/` after first run, so re-rendering a figure does not require re-clustering.
- All clustering uses `sklearn.cluster.AgglomerativeClustering` with `metric='euclidean'`. Ward linkage operates on squared Euclidean distance internally; the other linkages use raw Euclidean distance.
- Figures are exported as both PDF (vector, primary) and PNG (600 dpi, fallback) to `/content/figures/supplement/`.


## Section 1 — Imports & Data Loading

In [ ]:
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgb

from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import linkage, leaves_list

from joblib import Parallel, delayed
from tqdm import tqdm

# Journal-quality global settings (apply to every figure exported from this notebook)
mpl.rcParams['pdf.fonttype']       = 42        # TrueType, not Type 3
mpl.rcParams['ps.fonttype']        = 42
mpl.rcParams['font.family']        = 'sans-serif'
mpl.rcParams['font.sans-serif']    = ['Arial', 'Helvetica', 'DejaVu Sans']
mpl.rcParams['savefig.bbox']       = 'tight'
mpl.rcParams['savefig.pad_inches'] = 0.05

In [ ]:
# Load trajectory data
with open('/content/patient_vectors.pkl', 'rb') as f:
    patient_vectors = pickle.load(f)

with open('/content/patient_bins.pkl', 'rb') as f:
    patient_bins = pickle.load(f)

# Build the clustering input matrix (sorted patient ids -> X)
patient_ids = sorted(patient_vectors.keys())
X = np.array([patient_vectors[pid] for pid in patient_ids])

print(f'Patients: {len(patient_ids):,}')
print(f'Vector dimensionality: {X.shape[1]}')

# Output directories
CACHE_DIR  = Path('/content/cache')
FIGURE_DIR = Path('/content/figures/supplement')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Cache:   {CACHE_DIR}')
print(f'Figures: {FIGURE_DIR}')

## Section 2 — Helper Functions

Three groups of helpers:

1. **Validity metrics** — `compute_wcss` and gap-statistic helpers, copied from `Clustering.ipynb` so this notebook is self-contained.
2. **Heatmap rendering** — refactored `plot_heatmap_on_ax` that paints a patient-medication heatmap onto a given matplotlib axis (the original `create_heatmap` made its own figure, which doesn't compose into a 2×2 grid).
3. **Cluster ordering** — `order_clusters_for_display` runs Ward's linkage on the cluster centroids and returns an optimal leaf ordering. This is purely a visual convenience — adjacent clusters in the heatmap will look similar — and matches the approach used in `PatientHeatMapCreation.ipynb`.

In [ ]:
# --- Validity metrics ---
def compute_wcss(X, labels):
    """Within-cluster sum of squared distances to centroids."""
    wcss = 0.0
    for cluster_id in np.unique(labels):
        cluster_points = X[labels == cluster_id]
        centroid = cluster_points.mean(axis=0)
        wcss += np.sum((cluster_points - centroid) ** 2)
    return wcss


def generate_reference_data(X, B, seed=0):
    """B uniform-random reference datasets bounded by the per-feature min/max of X."""
    rng = np.random.default_rng(seed)
    mins = X.min(axis=0)
    maxs = X.max(axis=0)
    return [rng.uniform(low=mins, high=maxs, size=X.shape) for _ in range(B)]


def _cluster_reference(k, ref_data, b_idx):
    """Cluster a single reference dataset and return its log(WCSS)."""
    model = AgglomerativeClustering(n_clusters=k, linkage='ward', metric='euclidean')
    labels = model.fit_predict(ref_data)
    return (k, b_idx, np.log(compute_wcss(ref_data, labels)))

In [ ]:
# --- Heatmap rendering ---
PRESCRIPTION_COLORS = {
    'SUL':       '#FFC0CB',  # Light pink
    'SGLT2':     '#008000',  # Green
    'Insulin':   '#FF0000',  # Red
    'MET':       '#0000FF',  # Blue
    'DPP-4':     '#8B4513',  # Brown
    'GLP-1':     '#FFDB58',  # Mustard
    'GIP/GLP-1': '#40E0D0',  # Turquoise
    'TZD':       '#FF8C00',  # Bright orange
    'Other':     '#000000',  # Black
    'nothing':   '#FFFFFF',  # White
}

def mix_colors(colors):
    rgb = np.array([to_rgb(PRESCRIPTION_COLORS[c]) for c in colors if c in PRESCRIPTION_COLORS])
    if len(rgb) == 0:
        return np.array(to_rgb(PRESCRIPTION_COLORS['nothing']))
    return np.mean(rgb, axis=0).astype(np.float32)


def plot_heatmap_on_ax(ax, patient_bins, cluster_patient_ids, title_str,
                        cluster_ordering=None, ytick_step=2000):
    """Paint a patient-medication heatmap onto a given axis (subplot-friendly).

    Parameters
    ----------
    ax : matplotlib axis
    patient_bins : dict[patient_id -> list of 12 sets]
    cluster_patient_ids : dict[cluster_id -> list of patient_ids]
    title_str : str
    cluster_ordering : optional list of cluster_ids in the order they should appear top->bottom
    ytick_step : int, spacing between y-axis ticks (patient count)
    """
    ordering = cluster_ordering if cluster_ordering is not None else list(cluster_patient_ids.keys())

    sorted_patients = []
    cluster_boundaries = []
    start_idx = 0
    for cluster_id in ordering:
        patient_list = cluster_patient_ids[cluster_id]
        sorted_patients.extend(patient_list)
        start_idx += len(patient_list)
        cluster_boundaries.append(start_idx)

    num_patients = len(sorted_patients)
    num_bins = 12

    heatmap_data = np.ones((num_patients, num_bins, 3), dtype=np.float32)
    for i, pid in enumerate(sorted_patients):
        for j, prescriptions in enumerate(patient_bins[pid]):
            heatmap_data[i, j] = mix_colors(prescriptions)

    ax.imshow(heatmap_data, aspect='auto', interpolation='none')

    # Vertical separators between half-year bins
    for bin_idx in range(1, num_bins):
        ax.axvline(x=bin_idx - 0.5, color='black', linewidth=0.6, linestyle='dashed', alpha=0.6)

    # Horizontal cluster boundaries (skip the implicit boundary at 0)
    for boundary in cluster_boundaries[:-1]:
        if boundary > 0:
            ax.axhline(y=boundary - 0.5, color='black', linewidth=0.8)

    # Y-axis: patient count ticks
    y_ticks = list(range(ytick_step, num_patients + 1, ytick_step))
    ax.set_yticks(y_ticks)
    ax.set_yticklabels([f'{t:,}' for t in y_ticks], fontsize=8)
    ax.set_ylabel('Patient Count', fontsize=9)

    # X-axis: half-year labels
    x_labels = [f'{year}-H{half}' for year in range(2019, 2025) for half in (1, 2)]
    ax.set_xticks(range(num_bins))
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=8)

    ax.set_title(title_str, fontsize=11, fontweight='bold', pad=12)


def make_legend_handles():
    """Shared legend handles for any of the multi-panel heatmap figures."""
    return [mpatches.Patch(color=color, label=drug)
            for drug, color in PRESCRIPTION_COLORS.items() if drug != 'nothing']

In [ ]:
# --- Cluster ordering & post-processing ---
def labels_to_sorted_cluster_dict(labels, patient_ids):
    """Group patient_ids by label and re-key from 1 in descending order of size."""
    raw = {}
    for pid, lab in zip(patient_ids, labels):
        raw.setdefault(int(lab), []).append(pid)
    sorted_clusters = sorted(raw.items(), key=lambda x: len(x[1]), reverse=True)
    return {i + 1: pids for i, (_, pids) in enumerate(sorted_clusters)}


def order_clusters_for_display(cluster_patient_ids, patient_vectors):
    """Order clusters so that adjacent rows in the heatmap are most similar.

    Builds each cluster's centroid in the trajectory feature space, runs Ward's
    linkage on the centroids, and returns the optimal leaf order from
    scipy.cluster.hierarchy.leaves_list.
    """
    centroids = []
    cids = []
    for cid, pids in cluster_patient_ids.items():
        vecs = [patient_vectors[pid] for pid in pids]
        if vecs:
            centroids.append(np.mean(vecs, axis=0))
            cids.append(cid)
    centroids = np.array(centroids)
    Z = linkage(centroids, method='ward')
    leaf_order = leaves_list(Z)
    return [cids[i] for i in leaf_order]

In [ ]:
# --- Figure export helper ---
def save_figure(fig, stem, dpi=600):
    """Save fig as PDF (vector) and PNG (raster) with consistent settings."""
    pdf_path = FIGURE_DIR / f'{stem}.pdf'
    png_path = FIGURE_DIR / f'{stem}.png'
    fig.savefig(pdf_path)
    fig.savefig(png_path, dpi=dpi)
    print(f'  -> {pdf_path}')
    print(f'  -> {png_path}')

## Section 3 — Figure A: Internal Validity Diagnostics for k

Two side-by-side panels: WCSS (elbow) and gap statistic across k = 2..50, both using Ward's linkage.

**Why both metrics.** The elbow method is visually intuitive but subjective; the gap statistic provides a more formal selection criterion. Together they triangulate the choice of k.

**Computational note.** This section is the bottleneck. Computing the gap statistic with B = 10 reference datasets across 49 k-values requires 490 reference clusterings on top of 49 real-data clusterings. Results are cached after first run.

In [ ]:
# --- Compute WCSS and gap-statistic curves ---
K_VALUES = list(range(2, 51))   # k = 2..50
B_REF    = 10                   # number of reference datasets for gap statistic
CACHE_FILE = CACHE_DIR / 'wcss_gap_k2_50_B10.pkl'

if CACHE_FILE.exists():
    with open(CACHE_FILE, 'rb') as f:
        cached = pickle.load(f)
    wcss_scores = cached['wcss']
    gap_values  = cached['gap']
    print(f'Loaded cached WCSS and gap values from {CACHE_FILE}')
else:
    print(f'Computing WCSS and gap statistic for k = {K_VALUES[0]}..{K_VALUES[-1]} '
          f'with B = {B_REF} reference datasets...')

    # 1) Real-data WCSS for each k
    wcss_scores = []
    real_models = {}
    for k in tqdm(K_VALUES, desc='Real WCSS'):
        model = AgglomerativeClustering(n_clusters=k, linkage='ward', metric='euclidean')
        labels = model.fit_predict(X)
        wcss_scores.append(compute_wcss(X, labels))
        real_models[k] = labels

    # 2) Reference WCSS (parallel)
    ref_data_list = generate_reference_data(X, B_REF, seed=0)
    ref_tasks = [(k, ref, b) for b, ref in enumerate(ref_data_list) for k in K_VALUES]
    ref_results = Parallel(n_jobs=-1)(
        delayed(_cluster_reference)(k, ref, b) for (k, ref, b) in tqdm(ref_tasks, desc='Reference WCSS')
    )
    ref_log_wcss = {(k, b): logw for (k, b, logw) in ref_results}

    # 3) Gap_k = mean(log(W_ref)) - log(W_real)
    gap_values = []
    for k, w_real in zip(K_VALUES, wcss_scores):
        ref_logs = [ref_log_wcss[(k, b)] for b in range(B_REF)]
        gap_values.append(np.mean(ref_logs) - np.log(w_real))

    with open(CACHE_FILE, 'wb') as f:
        pickle.dump({'wcss': wcss_scores, 'gap': gap_values, 'k': K_VALUES}, f)
    print(f'Cached results to {CACHE_FILE}')

In [ ]:
# --- Render Figure A ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# Panel 1: WCSS (elbow)
ax = axes[0]
ax.plot(K_VALUES, wcss_scores, marker='o', markersize=4, linewidth=1.4, color='#1f77b4')
ax.axvline(x=40, color='red', linestyle='--', linewidth=1, alpha=0.7, label='k = 40 (selected)')
ax.set_xlabel('Number of Clusters (k)', fontsize=10)
ax.set_ylabel('WCSS', fontsize=10)
ax.set_title('A. Elbow Method (WCSS)', fontsize=11, fontweight='bold', pad=10)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9, frameon=False)
ax.tick_params(axis='both', labelsize=9)

# Panel 2: Gap statistic
ax = axes[1]
ax.plot(K_VALUES, gap_values, marker='o', markersize=4, linewidth=1.4, color='#2ca02c')
ax.axvline(x=40, color='red', linestyle='--', linewidth=1, alpha=0.7, label='k = 40 (selected)')
ax.set_xlabel('Number of Clusters (k)', fontsize=10)
ax.set_ylabel('Gap Statistic', fontsize=10)
ax.set_title('B. Gap Statistic', fontsize=11, fontweight='bold', pad=10)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9, frameon=False)
ax.tick_params(axis='both', labelsize=9)

plt.tight_layout()
save_figure(fig, 'figA_wcss_gap_statistic')
plt.show()

### Layout Preview — Heatmap Figures (B & C)

Quick mockup of the 2×2 heatmap grid layout using random RGB data, so you can confirm the suptitle size, top-positioned legend, and inter-row spacing before committing to the full clustering runs (which take several minutes each).

The mockup is **layout-only** — the colored noise inside each panel is meaningless. The four panel titles, axis labels, suptitle styling, and legend position are exactly what Figures B and C will use.

In [ ]:
# --- Layout mockup: 2x2 grid with random RGB data, real styling ---
# Render is near-instant; no clustering involved.

def render_layout_mockup(panel_titles, suptitle, n_fake_patients=1000, seed=0):
    """Render a 2x2 grid mockup with the exact spacing/title/legend the real
    figures will use. Useful for iterating on layout before running the full
    clustering pipeline."""
    rng = np.random.default_rng(seed)
    num_bins = 12
    x_labels = [f'{year}-H{half}' for year in range(2019, 2025) for half in (1, 2)]

    fig, axes = plt.subplots(2, 2, figsize=(13, 15))
    fig.subplots_adjust(top=0.86, bottom=0.05, left=0.07, right=0.95,
                        hspace=0.38, wspace=0.20)

    for ax, title in zip(axes.flatten(), panel_titles):
        fake = rng.random((n_fake_patients, num_bins, 3)).astype(np.float32)
        ax.imshow(fake, aspect='auto', interpolation='none')

        for bin_idx in range(1, num_bins):
            ax.axvline(x=bin_idx - 0.5, color='black', linewidth=0.6,
                       linestyle='dashed', alpha=0.6)

        y_ticks = list(range(200, n_fake_patients + 1, 200))
        ax.set_yticks(y_ticks)
        ax.set_yticklabels([f'{t:,}' for t in y_ticks], fontsize=8)
        ax.set_ylabel('Patient Count', fontsize=9)

        ax.set_xticks(range(num_bins))
        ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=8)
        ax.set_title(title, fontsize=11, fontweight='bold', pad=12)

    # Suptitle: bigger and bold
    fig.suptitle(suptitle, fontsize=18, fontweight='bold', y=0.975)

    # Legend at TOP, just under the suptitle
    fig.legend(handles=make_legend_handles(),
               loc='upper center', bbox_to_anchor=(0.5, 0.935),
               ncol=9, fontsize=9, frameon=False,
               title='Medication Class', title_fontsize=10)

    plt.show()
    return fig


# Preview Figure B layout
_ = render_layout_mockup(
    panel_titles=['k = 20', 'k = 30', 'k = 40', 'k = 50'],
    suptitle="Ward's Linkage Clustering at Varying k"
)

# Preview Figure C layout
_ = render_layout_mockup(
    panel_titles=['Single Linkage', 'Complete Linkage', 'Average Linkage', "Ward's Linkage"],
    suptitle='Comparison of Agglomerative Linkage Methods at k = 40',
    seed=1
)

## Section 4 — Figure B: Ward's Linkage at k ∈ {20, 30, 40, 50}

A 2×2 grid of patient-medication heatmaps showing how the partition structure evolves with k. Each panel is the same 18,652 patients, ordered by an intra-figure leaf ordering (Ward on cluster centroids) so that adjacent clusters are visually similar.

In [ ]:
# --- Therapy-category ordering (consistent across panels) ---

# Top-to-bottom display order in the heatmap
THERAPY_DISPLAY_ORDER = [
    'GIP/GLP-1',     # PI label: "DUAL GIP/GLP-1" (dual incretin receptor agonist)
    'GLP-1',
    'SGLT2',
    'DPP-4',
    'TZD',
    'MET',
    'Insulin',
    'Early Dropout',
]

# Classification priority — specialty drugs first, then Insulin, then MET as backbone
THERAPY_CLASSIFICATION_PRIORITY = ['GIP/GLP-1', 'GLP-1', 'SGLT2', 'DPP-4', 'TZD', 'Insulin', 'MET']

PREVALENCE_THRESHOLD = 0.20  # fraction of patient-bins on a drug for it to "claim" the cluster
DROPOUT_NOTHING_THRESHOLD = 0.70  # fraction of empty bins in second half to flag as dropout

def classify_cluster(cluster_pids, patient_bins):
    n_patients = len(cluster_pids)
    n_bins = 12
    drug_classes = ['MET', 'SUL', 'SGLT2', 'GLP-1', 'GIP/GLP-1', 'Insulin', 'DPP-4', 'TZD', 'Other']
    drug_counts = {d: np.zeros(n_bins) for d in drug_classes}
    nothing_counts = np.zeros(n_bins)

    for pid in cluster_pids:
        for bin_idx, bin_set in enumerate(patient_bins[pid]):
            for drug in bin_set:
                if drug in drug_counts:
                    drug_counts[drug][bin_idx] += 1
                elif drug == 'nothing':
                    nothing_counts[bin_idx] += 1

    avg_prev = {d: np.mean(drug_counts[d]) / n_patients for d in drug_classes}
    second_half_nothing = np.mean(nothing_counts[6:]) / n_patients

    # Early Dropout if the second half (2022-2024) is mostly empty
    if second_half_nothing > DROPOUT_NOTHING_THRESHOLD:
        return 'Early Dropout'

    # Walk classification priority list
    for drug in THERAPY_CLASSIFICATION_PRIORITY:
        if avg_prev[drug] >= PREVALENCE_THRESHOLD:
            return drug

    # Fallback: most prevalent class if anything has meaningful presence, else Early Dropout
    top_drug, top_prev = max(avg_prev.items(), key=lambda x: x[1])
    return top_drug if top_prev > 0.05 else 'Early Dropout'


def order_clusters_by_therapy_category(cluster_patient_ids, patient_bins,
                                        display_order=THERAPY_DISPLAY_ORDER):
    classified = []
    for cid, pids in cluster_patient_ids.items():
        category = classify_cluster(pids, patient_bins)
        classified.append((cid, category, len(pids)))

    cat_idx = {cat: i for i, cat in enumerate(display_order)}
    classified.sort(key=lambda x: (cat_idx.get(x[1], 99), -x[2]))

    ordering = [cid for cid, _, _ in classified]
    # Return both the ordering and the classifications, so the diagnostic print below can use them
    classifications = {cid: cat for cid, cat, _ in classified}
    return ordering, classifications

In [ ]:
# --- Run Ward clustering at each target k ---
K_TARGETS = [20, 30, 40, 50]
ward_results = {}  # k -> (cluster_patient_ids dict, ordering list)

for k in K_TARGETS:
    print(f'Clustering with Ward, k = {k}...')
    model = AgglomerativeClustering(n_clusters=k, linkage='ward', metric='euclidean')
    labels = model.fit_predict(X)
    cpi = labels_to_sorted_cluster_dict(labels, patient_ids)

    ordering, classifications = order_clusters_by_therapy_category(cpi, patient_bins)
    ward_results[k] = (cpi, ordering)

    # Diagnostic: category breakdown
    from collections import Counter
    cat_counts = Counter(classifications.values())
    print(f'  -> {k} clusters | sizes: min={min(len(v) for v in cpi.values())}, '
          f'max={max(len(v) for v in cpi.values())}')
    for cat in THERAPY_DISPLAY_ORDER:
        n_clusters = cat_counts.get(cat, 0)
        n_patients = sum(len(cpi[cid]) for cid, c in classifications.items() if c == cat)
        if n_clusters > 0:
            print(f'     {cat:15s}: {n_clusters} cluster(s), {n_patients:,} patients')

In [ ]:
# --- Render Figure B ---
# 2x2 grid with bigger bold suptitle, legend at top under the title, reduced hspace.
fig, axes = plt.subplots(2, 2, figsize=(13, 15))
fig.subplots_adjust(top=0.86, bottom=0.05, left=0.07, right=0.95,
                    hspace=0.38, wspace=0.20)

for ax, k in zip(axes.flatten(), K_TARGETS):
    cpi, ordering = ward_results[k]
    plot_heatmap_on_ax(ax, patient_bins, cpi,
                       title_str=f'k = {k}',
                       cluster_ordering=ordering,
                       ytick_step=2000)

# Suptitle: bigger and bold
fig.suptitle("Ward's Linkage Clustering at Varying k",
             fontsize=18, fontweight='bold', y=0.975)

# Shared legend at TOP, just under the suptitle
fig.legend(handles=make_legend_handles(),
           loc='upper center', bbox_to_anchor=(0.5, 0.935),
           ncol=9, fontsize=9, frameon=False,
           title='Medication Class', title_fontsize=10)

save_figure(fig, 'figB_ward_k_comparison')
plt.show()

## Section 5 — Figure C: Linkage-Method Comparison at k = 40

A 2×2 grid showing the four classical agglomerative-linkage methods at the same k = 40. All use Euclidean distance.

**Expected reading.** Single linkage typically produces a "chaining" effect on binary data — one large cluster dominated by patients with sparse trajectories, and many tiny outlier clusters. Complete and average linkage are intermediate. Ward's linkage produces the most balanced and clinically interpretable partition, which is why it was selected for the primary analysis.

In [ ]:
# --- Run each linkage method at k = 40 ---
LINKAGE_METHODS = ['single', 'complete', 'average', 'ward']
LINKAGE_LABELS  = {
    'single':   'Single Linkage',
    'complete': 'Complete Linkage',
    'average':  'Average Linkage',
    'ward':     "Ward's Linkage",
}
K_FIXED = 40

linkage_results = {}  # method -> (cluster_patient_ids, ordering)

for method in LINKAGE_METHODS:
    print(f'Clustering with {method} linkage at k = {K_FIXED}...')
    model = AgglomerativeClustering(n_clusters=K_FIXED, linkage=method, metric='euclidean')
    labels = model.fit_predict(X)
    cpi = labels_to_sorted_cluster_dict(labels, patient_ids)
    ordering = order_clusters_for_display(cpi, patient_vectors)
    linkage_results[method] = (cpi, ordering)
    sizes = sorted([len(v) for v in cpi.values()], reverse=True)
    print(f'  -> sizes (top 5): {sizes[:5]} | (bottom 5): {sizes[-5:]}')

In [ ]:
# --- Render Figure C ---
# Same layout treatment as Figure B: bigger bold suptitle, legend at top, reduced hspace.
fig, axes = plt.subplots(2, 2, figsize=(13, 15))
fig.subplots_adjust(top=0.86, bottom=0.05, left=0.07, right=0.95,
                    hspace=0.38, wspace=0.20)

for ax, method in zip(axes.flatten(), LINKAGE_METHODS):
    cpi, ordering = linkage_results[method]
    plot_heatmap_on_ax(ax, patient_bins, cpi,
                       title_str=LINKAGE_LABELS[method],
                       cluster_ordering=ordering,
                       ytick_step=2000)

# Suptitle: bigger and bold
fig.suptitle('Comparison of Agglomerative Linkage Methods at k = 40',
             fontsize=18, fontweight='bold', y=0.975)

# Shared legend at TOP, just under the suptitle
fig.legend(handles=make_legend_handles(),
           loc='upper center', bbox_to_anchor=(0.5, 0.935),
           ncol=9, fontsize=9, frameon=False,
           title='Medication Class', title_fontsize=10)

save_figure(fig, 'figC_linkage_comparison')
plt.show()

## Section 6 — Summary

Three supplementary figures have been exported to `/content/figures/supplement/`:

- **`figA_wcss_gap_statistic.{pdf,png}`** — WCSS elbow + gap statistic across k = 2..50
- **`figB_ward_k_comparison.{pdf,png}`** — Ward's linkage at k = 20, 30, 40, 50
- **`figC_linkage_comparison.{pdf,png}`** — Single, complete, average, Ward linkage at k = 40

These can be inserted into the supplementary materials as Figures S? through S? (numbering depending on what's already in the supplement). The PDF versions are vector and should be the primary submission files; PNGs are 600 dpi fallbacks.

**Suggested supplementary captions:**

> **Figure S?.** Internal-validity diagnostics for selecting the number of clusters (k). (A) Within-cluster sum of squares (WCSS) by k, computed using Ward's linkage on 18,652 patient trajectories. The curve flattens gradually after k ≈ 25–35, with no sharp elbow. (B) Gap statistic by k, computed against B = 10 uniform-random reference datasets. The selected value k = 40 is marked in red.

> **Figure S?.** Ward's linkage clustering at four values of k (20, 30, 40, 50). Each row of each heatmap is a single patient; columns are six-month bins from 2019-H1 to 2024-H2; colors encode medication class. Horizontal black lines separate clusters. Cluster ordering within each panel is determined by Ward's linkage on cluster centroids for visual coherence.

> **Figure S?.** Comparison of agglomerative linkage methods at k = 40 on the same 18,652 patient trajectories. Single linkage exhibits the expected chaining effect (one dominant cluster, many singletons). Complete and average linkage are intermediate. Ward's linkage produces the most balanced and clinically interpretable partition and was selected for the primary analysis.
